# Part 3 — TESSERA for all 2,433 patches (2019 and 2018)

**Goal:** `pj_vmi/processed/` holds, for each year, `tessera_<year>_q.npy` (2433, 128, 128, 128) int8, `tessera_<year>_scale.npy` (2433, 128, 128) float32 (embedding = `q * scale`, NaN scale = no data), `qa_tessera_<year>.csv` and `tessera_<year>_info.json`.

**Needs Part 1 on the Drive** (`processed/`). **Runtime:** CPU; keep the tab open. **Time:** about 25 min per year for the download (pilot estimate), then a few minutes to merge and 5–15 min to copy to the Drive. **Drive space:** about 10.5 GB for the merged files; the shards (about 10.5 GB) are deleted at the end of the notebook.

**If the runtime disconnects:** run steps 1–3 again, then the cell that was running. The download resumes: finished shards (100 patches each) are skipped.

## 1. Runtime, Drive and configuration

In [ ]:
import sys, shutil, platform
from pathlib import Path
import json, numpy as np, pandas as pd
print("Python", platform.python_version())
assert sys.version_info >= (3, 12), "geotessera needs Python >= 3.12"
print(f"free space in /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

from google.colab import drive
drive.mount("/content/drive")
!df -h /content/drive | tail -1

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")

PROC = Path("/content/processed")       # merged arrays are written here first
PROC_DRIVE = DRIVE / "processed"
SHARDS = DRIVE / "shards"               # resumable download, deleted at the end
YEARS = [2019, 2018]

## 2. Code

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt

## 3. Part 1 outputs

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "instances.npy"]:
    shutil.copy(PROC_DRIVE / f, PROC / f)
meta = pd.read_csv(PROC / "meta.csv", index_col="idx")
print(len(meta), "patches")

## 4. Download, one year per cell
Each line `[part k/25]` is one shard of 100 patches written to the Drive, with the number of errors and the remaining time.

**Check:** the cell ends with `done` and every shard line says `errors: 0`. If some shard reports errors, run the extra cell below it (`--retry-errors` fetches only the failed patches again).

In [ ]:
!python scripts/fetch_tessera.py --processed "{PROC}" --year 2019 --shard-dir "{SHARDS}/tessera_2019" --workers 6

In [ ]:
# only if some shard of 2019 reported errors
!python scripts/fetch_tessera.py --processed "{PROC}" --year 2019 --shard-dir "{SHARDS}/tessera_2019" --workers 6 --retry-errors

In [ ]:
!python scripts/fetch_tessera.py --processed "{PROC}" --year 2018 --shard-dir "{SHARDS}/tessera_2018" --workers 6

In [ ]:
# only if some shard of 2018 reported errors
!python scripts/fetch_tessera.py --processed "{PROC}" --year 2018 --shard-dir "{SHARDS}/tessera_2018" --workers 6 --retry-errors

## 5. Merge the shards
**Check**, for each year: `25 shards, 2433/2433 patches`, `missing: 0, duplicated: 0, errors: 0`. Write down the other lines (patches with no-data pixels, best shift other than (0, 0), median ratio): they go into the data section of the report.

In [ ]:
for y in YEARS:
    print(f"===== {y}")
    !python scripts/merge_shards.py --shard-dir "{SHARDS}/tessera_{y}" --processed "{PROC}" --name tessera_{y}

## 6. Flagged patches in both years
A patch whose best shift is not (0, 0) in **both** years points to a real grid problem; a flag in one year only is more likely noise (few or very small parcels, low ratio contrast). The table shows the ratio at (0, 0) and at the best shift, with the number of parcels and the parcel cover of the patch.

In [ ]:
pd.set_option("display.width", 200)
flags = {}
for y in YEARS:
    qa = pd.read_csv(PROC / f"qa_tessera_{y}.csv", index_col="idx")
    shifted = (qa.best_dx.fillna(0) != 0) | (qa.best_dy.fillna(0) != 0)
    flags[y] = set(qa.index[shifted | (qa.nodata_frac > 0)])
    print(f"{y}: shift != (0, 0): {int(shifted.sum())}, no-data > 0: {int((qa.nodata_frac > 0).sum())}, "
          f"median ratio {qa['ratio_+0_+0'].median():.3f}")
both = sorted(flags[2019] & flags[2018])
print(f"flagged in both years: {len(both)}; in one year only: {len(flags[2019] ^ flags[2018])}")

rows = []
for y in YEARS:
    qa = pd.read_csv(PROC / f"qa_tessera_{y}.csv", index_col="idx")
    for i in sorted(flags[2019] | flags[2018]):
        r = qa.loc[i]
        best = f"ratio_{int(r.best_dx):+d}_{int(r.best_dy):+d}" if pd.notna(r.best_dx) else "ratio_+0_+0"
        rows.append({"year": y, "ID_PATCH": int(r.ID_PATCH), "TILE": meta.loc[i, "TILE"],
                     "N_Parcel": int(meta.loc[i, "N_Parcel"]), "Parcel_Cover": round(meta.loc[i, "Parcel_Cover"], 2),
                     "best": (r.best_dx, r.best_dy), "ratio_0": round(r["ratio_+0_+0"], 3),
                     "ratio_best": round(r[best], 3), "nodata": round(r.nodata_frac, 3), "in_both": i in both})
print(pd.DataFrame(rows).sort_values(["ID_PATCH", "year"]).to_string(index=False) if rows else "no flagged patch")

## 7. Copy to the Drive
**Check:** every line ends with `ok=True`.

In [ ]:
for y in YEARS:
    for f in [f"tessera_{y}_q.npy", f"tessera_{y}_scale.npy", f"qa_tessera_{y}.csv", f"tessera_{y}_info.json"]:
        shutil.copy(PROC / f, PROC_DRIVE / f)
        same = (PROC_DRIVE / f).stat().st_size == (PROC / f).stat().st_size
        print(f"{f:28s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={same}")

## 8. Delete the shards
The cell deletes the shards of a year only if its merged files are on the Drive with the same size as the local ones. It frees about 10.5 GB.

In [ ]:
for y in YEARS:
    files = [f"tessera_{y}_q.npy", f"tessera_{y}_scale.npy"]
    ok = all((PROC_DRIVE / f).exists() and (PROC_DRIVE / f).stat().st_size == (PROC / f).stat().st_size
             for f in files)
    if ok and (SHARDS / f"tessera_{y}").exists():
        shutil.rmtree(SHARDS / f"tessera_{y}")
        print("deleted", SHARDS / f"tessera_{y}")
    else:
        print("kept", SHARDS / f"tessera_{y}", "" if ok else "(merged files not verified on the Drive)")
!df -h /content/drive | tail -1